# Session: Time Series Forecasting

<table cellpadding="10" cellspacing="0" border="0" width="100%"><tr>
<td bgcolor="#ED1C24" width="4"></td>
<td bgcolor="#fce4ec">
<font color="#c62828"><b>Program:</b></font> Vishlesan i-Hub IIT Patna x Masai School — AIM (AI &amp; Machine Learning)<br>
<font color="#c62828"><b>Session ID:</b></font> TS.1 | <b>Week:</b> 16 | <b>Module:</b> Module 2 — Core ML &amp; Engineering<br>
<font color="#c62828"><b>Prerequisites:</b></font> Linear regression, feature engineering basics, pandas date handling<br>
<font color="#c62828"><b>Estimated completion time:</b></font> 75–90 minutes (self-paced)
</td>
</tr></table>

---

Vishlesan i-Hub IIT Patna × Masai School

## Learning Objectives

By the end of this notebook you will be able to:

1. Decompose a time series into trend, seasonality, and residual components using additive and multiplicative models.
2. Test for stationarity with the Augmented Dickey-Fuller test and apply differencing to achieve it.
3. Interpret ACF and PACF plots to manually select ARIMA(p, d, q) orders.
4. Select ARIMA hyperparameters systematically via AIC grid search.
5. Engineer calendar and Fourier time features to encode seasonality without external libraries.
6. Build a Prophet-style linear trend + Fourier seasonality model and understand what Prophet does under the hood.
7. Evaluate forecasting models honestly using walk-forward (expanding-window) validation.
8. Compare models on MAE, RMSE, and MAPE on identical out-of-sample windows.

## Why this session matters — the Flipkart Big Billion Days inventory crisis

Every year, Flipkart and Amazon India run 4–7 day mega-sale events. The teams that manage warehouse stock have exactly **one job**: stock the right number of units of each SKU so that popular items never go out of stock (lost revenue) and slow items are not over-stocked (capital locked up in warehouse space).

That job is a **time series forecasting problem**.

In 2019, several category teams at major e-commerce platforms reported post-mortem losses from both directions — stockouts on electronics and overstock on fashion. The root causes traced back to three forecasting failures, all of which are the subject of this notebook:

1. **Ignoring seasonality structure.** Sales patterns for electronics peak in Q4 with a sharp weekly cycle (payday weeks spike). A model fit on raw, non-decomposed data treats these spikes as noise and systematically under-predicts peak demand.
2. **Non-stationary data fed directly into ARIMA.** ARIMA requires stationarity. Teams that skip the ADF test and feed a trending series directly into `ARIMA(1,0,1)` get coefficients that chase the trend instead of modeling the true dynamics — and forecasts that drift wildly outside the sale window.
3. **Holdout evaluation instead of walk-forward.** A single train/test split hides the fact that ARIMA's forecast degrades rapidly beyond 2–3 steps. Walk-forward validation exposes this degradation before the model goes live.

Every concept we code today is one of the diagnostics that, applied honestly, would have caught these problems in backtesting.

## Setup & Imports

In [1]:
# Install / upgrade only what Colab may not have pre-installed
# We do NOT force-upgrade numpy/pandas/scipy/sklearn — Colab's versions are fine.
!pip install -q "statsmodels>=0.14" "plotly>=5.20"

In [2]:
import os
import warnings
import itertools
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats

os.makedirs("outputs", exist_ok=True)

SEED = 42
np.random.seed(SEED)

MASAI_RED = "#ED1C24"
BLUE      = "#1565C0"
GREEN     = "#2E7D32"
ORANGE    = "#E65100"
PURPLE    = "#7C4DFF"
GREY      = "#95A5A6"

import statsmodels
import sklearn
print(f"Setup complete. statsmodels {statsmodels.__version__} | sklearn {sklearn.__version__}")

Setup complete. statsmodels 0.14.6 | sklearn 1.9.0


## The `box()` helper for styled callouts

In [3]:
from IPython.display import HTML, display

_BOX_STYLES = {
    "definition": ("#448aff", "#e3f2fd", "#1565c0"),
    "tip":        ("#00c853", "#e8f5e9", "#2e7d32"),
    "warning":    ("#ff9100", "#fff3e0", "#e65100"),
    "danger":     ("#ff1744", "#fce4ec", "#c62828"),
    "math":       ("#7c4dff", "#ede7f6", "#4527a0"),
    "output":     ("#00b8d4", "#e0f7fa", "#006064"),
    "industry":   ("#009688", "#e0f2f1", "#004d40"),
}

def box(kind, title, content):
    border, bg, title_clr = _BOX_STYLES[kind]
    display(HTML(f"""
    <div style=\"margin:12px 0; color: gray; padding:12px 16px; border-left:4px solid {border};
                background-color:{bg}; border-radius:4px;\">
    <strong style=\"color:{title_clr};\">{title}</strong><br>{content}
    </div>"""))

In [4]:
box("industry", "Three forecasting failures behind the Big Billion Days crisis",
    "<b>Non-decomposed data</b> treats seasonal spikes as noise — the model under-predicts peak demand.<br>"
    "<b>Non-stationary input to ARIMA</b> breaks the coefficient estimates — forecasts drift outside the sale window.<br>"
    "<b>Holdout-only evaluation</b> hides ARIMA degradation beyond 2-3 steps ahead — the model looks good in testing but fails live.<br><br>"
    "All three failure modes are diagnosed and fixed in this notebook.")

## Part 0 — Build a realistic synthetic e-commerce sales dataset

We simulate **3 years of weekly sales** for an electronics SKU on a large marketplace. The true data-generating process has four components:

- A **linear upward trend** (growing customer base)
- A **52-week annual seasonality** (Diwali / festive season spikes in Q4)
- A **weak 13-week quarterly cycle** (payday-week lifts)
- **Gaussian noise** scaled to ±10% of the mean

We know the truth, so we can check whether our models recover the right structure.

In [5]:
N      = 156          # 3 years × 52 weeks
FREQ   = "W-MON"
dates  = pd.date_range("2021-01-04", periods=N, freq=FREQ)
rng    = np.random.default_rng(SEED)
t      = np.arange(N)

# True components (what we want the model to find)
TREND_SLOPE     = 2.5          # Rs-equivalent units per week
TREND_INTERCEPT = 500
ANNUAL_AMP      = 80           # big festive-season wave
QUARTERLY_AMP   = 25           # smaller quarterly cycle
NOISE_STD       = 18

trend_comp    = TREND_INTERCEPT + TREND_SLOPE * t
seasonal_comp = (ANNUAL_AMP    * np.sin(2 * np.pi * t / 52)
               + QUARTERLY_AMP * np.sin(2 * np.pi * t / 13))
noise_comp    = rng.normal(0, NOISE_STD, N)
sales         = trend_comp + seasonal_comp + noise_comp

ts = pd.Series(sales.round(1), index=dates, name="weekly_sales")
print(f"Series shape : {ts.shape}")
print(f"Date range   : {ts.index[0].date()} → {ts.index[-1].date()}")
print(f"Mean sales   : {ts.mean():.1f}  |  Std: {ts.std():.1f}  |  Min: {ts.min():.1f}  |  Max: {ts.max():.1f}")
ts.head()

Series shape : (156,)
Date range   : 2021-01-04 → 2023-12-25
Mean sales   : 692.7  |  Std: 112.6  |  Min: 479.1  |  Max: 914.9


2021-01-04    505.5
2021-01-11    505.0
2021-01-18    558.2
2021-01-25    577.6
2021-02-01    535.4
Freq: W-MON, Name: weekly_sales, dtype: float64

In [6]:
box("definition", "Additive vs Multiplicative decomposition",
    "<b>Additive:</b> y(t) = Trend(t) + Seasonal(t) + Residual(t). "
    "Use when the seasonal swing is roughly constant in absolute size regardless of the trend level. "
    "Our electronics data has a fixed Rs swing each festive season — additive is correct.<br>"
    "<b>Multiplicative:</b> y(t) = Trend(t) × Seasonal(t) × Residual(t). "
    "Use when the seasonal swing <i>grows proportionally</i> with the level (e.g., percentage-based discounts). "
    "Equivalent to additive on log(y).")

In [8]:
# Quick look: raw series with true components overlaid
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ts.index, y=ts.values, mode="lines",
    line=dict(color=GREY, width=1.5), opacity=0.8,
    name="observed sales",
    hovertemplate="week=%{x|%Y-%m-%d}<br>sales=%{y:.1f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=ts.index, y=trend_comp, mode="lines",
    line=dict(color=MASAI_RED, width=2, dash="dash"),
    name="true trend",
    hovertemplate="week=%{x|%Y-%m-%d}<br>trend=%{y:.1f}<extra></extra>",
))
fig.update_layout(
    title="3 years of synthetic weekly electronics sales",
    xaxis_title="Date", yaxis_title="Units sold",
    paper_bgcolor="white", plot_bgcolor="white", width=950, height=400,
)
fig.show()

---

## Section 1 — Time Series Decomposition

**Teaching pattern in use: Visual Evidence Before Formal Definition.**

We plot the raw series, then pull it apart into trend, seasonality, and residual. Only after seeing the picture do we write down the equations.

In [25]:
# statsmodels additive decomposition with period = 52 (weekly annual cycle)
decomp = seasonal_decompose(ts, model="additive", period=52)

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    subplot_titles=("Observed", "Trend (centred moving average)", "Seasonality (period=52)", "Residual"),
    vertical_spacing=0.07,
)
components = [ts, decomp.trend, decomp.seasonal, decomp.resid]
colors     = [GREY, MASAI_RED, BLUE, ORANGE]
names      = ["observed", "trend", "seasonal", "residual"]

for i, (comp, col, nm) in enumerate(zip(components, colors, names), 1):
    fig.add_trace(go.Scatter(
        x=comp.index, y=comp.values, mode="lines",
        line=dict(color=col, width=1.8),
        name=nm,
        hovertemplate=f"week=%{{x|%Y-%m-%d}}<br>{nm}=%{{y:.2f}}<extra></extra>",
    ), row=i, col=1)

fig.update_layout(
    title="Additive decomposition: Observed = Trend + Seasonal + Residual",
    paper_bgcolor="white", plot_bgcolor="white",
    width=950, height=700, showlegend=False,
)
fig.show()

In [26]:
# How much variance does each component explain?
trend_clean   = decomp.trend.dropna()
seasonal_var  = decomp.seasonal.var()
trend_var     = trend_clean.var()
resid_var     = decomp.resid.dropna().var()
total_var     = ts.var()

box("output", "Variance decomposition (what each component explains)",
    f"<b>Trend</b> explains   {100*trend_var/total_var:.1f}% of total variance<br>"
    f"<b>Seasonal</b> explains {100*seasonal_var/total_var:.1f}% of total variance<br>"
    f"<b>Residual</b> explains {100*resid_var/total_var:.1f}% of total variance<br><br>"
    "In a well-decomposed series the residual should be the smallest component. "
    "If it dominates, either the period is wrong or the model form (additive vs multiplicative) is misspecified.")

In [27]:
box("math", "The additive decomposition identity",
    "y(t) = T(t) + S(t) + R(t)<br>"
    "where <b>T(t)</b> is estimated by a centred moving average of width = period, "
    "<b>S(t)</b> is the average residual after de-trending for each position within the period, "
    "and <b>R(t) = y(t) - T(t) - S(t)</b>.<br><br>"
    "seasonal_decompose() uses this classical approach. It needs at least 2 full periods of data. "
    "The first and last (period/2) observations have no trend estimate — that is the NaN band you see in the Trend panel.")

---

## Section 2 — Stationarity and the ADF Test

**Teaching pattern in use: Manual Derivation Before the Library Call.**

ARIMA requires stationarity — a constant mean, constant variance, and no trend. We test this formally with the Augmented Dickey-Fuller test, then apply differencing to fix it.

In [28]:
# ADF on the raw (clearly trending) series
adf_raw = adfuller(ts, autolag="AIC")
adf_stat_raw, p_raw, _, _, crit_raw, _ = adf_raw

print("ADF Test on RAW series")
print(f"  Test statistic : {adf_stat_raw:.4f}")
print(f"  p-value        : {p_raw:.4f}")
print(f"  Critical values: 1%={crit_raw['1%']:.3f}  5%={crit_raw['5%']:.3f}  10%={crit_raw['10%']:.3f}")
print(f"  Decision       : {'STATIONARY (reject H0)' if p_raw < 0.05 else 'NON-STATIONARY (fail to reject H0)'}")

ADF Test on RAW series
  Test statistic : -0.8489
  p-value        : 0.8043
  Critical values: 1%=-3.474  5%=-2.881  10%=-2.577
  Decision       : NON-STATIONARY (fail to reject H0)


In [29]:
box("math", "What the ADF test does",
    "<b>H₀ (null):</b> the series has a unit root — it is non-stationary (a random walk or has a trend).<br>"
    "<b>H₁ (alternative):</b> the series is stationary.<br>"
    "<b>Rule:</b> if the ADF statistic < critical value (5%), or p-value < 0.05, reject H₀ → stationary.<br><br>"
    "The <i>augmented</i> version adds lagged difference terms to absorb autocorrelation in the errors, "
    "so the test statistic has its tabulated distribution even when the series is serially correlated.")

In [30]:
# First difference: ts_diff[t] = ts[t] - ts[t-1]
ts_diff1 = ts.diff().dropna()
adf_diff1 = adfuller(ts_diff1, autolag="AIC")
adf_stat_d1, p_d1 = adf_diff1[0], adf_diff1[1]

print("ADF Test on FIRST-DIFFERENCED series")
print(f"  Test statistic : {adf_stat_d1:.4f}")
print(f"  p-value        : {p_d1:.4f}")
print(f"  Decision       : {'STATIONARY ✓' if p_d1 < 0.05 else 'Still non-stationary — try 2nd difference'}")

ADF Test on FIRST-DIFFERENCED series
  Test statistic : -7.9432
  p-value        : 0.0000
  Decision       : STATIONARY ✓


In [31]:
# Visual: raw vs first-differenced
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Original series (non-stationary — trending mean)",
                    "First-differenced series (week-over-week change)"),
    vertical_spacing=0.12)

fig.add_trace(go.Scatter(x=ts.index, y=ts.values, mode="lines",
    line=dict(color=BLUE, width=1.5), name="raw",
    hovertemplate="week=%{x|%Y-%m-%d}<br>sales=%{y:.1f}<extra></extra>"), row=1, col=1)

fig.add_trace(go.Scatter(x=ts_diff1.index, y=ts_diff1.values, mode="lines",
    line=dict(color=GREEN, width=1.5), name="Δ1",
    hovertemplate="week=%{x|%Y-%m-%d}<br>Δsales=%{y:.1f}<extra></extra>"), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color=MASAI_RED, row=2, col=1)

fig.update_yaxes(title_text="units sold", row=1, col=1)
fig.update_yaxes(title_text="week-over-week Δ", row=2, col=1)
fig.update_layout(title="Raw vs First-Differenced Series",
    paper_bgcolor="white", plot_bgcolor="white", width=950, height=500, showlegend=False)
fig.show()

In [32]:
box("output", "Why d=1 is the right integration order for this series",
    f"Raw series ADF p = {p_raw:.4f} → non-stationary (p > 0.05, fail to reject unit-root null).<br>"
    f"After one difference, ADF p = {p_d1:.4f} → stationary (p < 0.05).<br>"
    "This means <b>d = 1</b> in ARIMA(p, d, q). "
    "The intuition: differencing removes the trend by converting levels into changes. "
    "A weekly-sales series is almost always I(1) — one difference is almost always enough.")

---

## Section 3 — ACF and PACF: Reading the Lag Plots

**Teaching pattern in use: Manual Interpretation Before the Grid Search.**

The ACF and PACF plots are the traditional way to choose p and q in ARIMA. We read them by hand first, then confirm with an AIC grid search in Section 4.

In [37]:
# Compute ACF and PACF on the differenced (stationary) series
n_lags = 40
acf_vals, acf_ci  = acf(ts_diff1, nlags=n_lags, alpha=0.05, fft=True)
pacf_vals, pacf_ci = pacf(ts_diff1, nlags=n_lags, alpha=0.05)

lags = np.arange(n_lags + 1)
acf_lower  = acf_vals  - acf_ci[:, 0]
acf_upper  = acf_ci[:, 1] - acf_vals
pacf_lower = pacf_vals - pacf_ci[:, 0]
pacf_upper = pacf_ci[:, 1] - pacf_vals

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("ACF — Autocorrelation Function", "PACF — Partial Autocorrelation Function"))

for row_i, (vals, lower, upper, col_color, title_abbr) in enumerate([
    (acf_vals, acf_lower, acf_upper, BLUE, "ACF"),
    (pacf_vals, pacf_lower, pacf_upper, GREEN, "PACF"),
], 1):
    # Confidence band
    sig = 1.96 / np.sqrt(len(ts_diff1))
    fig.add_trace(go.Scatter(
        x=np.concatenate([lags, lags[::-1]]),
        y=np.concatenate([np.full(len(lags), sig), np.full(len(lags), -sig)[::-1]]),
        fill="toself", fillcolor="rgba(200,200,200,0.3)", line=dict(color="rgba(0,0,0,0)"),
        showlegend=False, hoverinfo="skip",
    ), row=1, col=row_i)
    # Stem lines
    for lag, val in zip(lags[1:], vals[1:]):
        fig.add_shape(type="line", x0=lag, x1=lag, y0=0, y1=val,
            line=dict(color=col_color, width=2), row=1, col=row_i)
    # Markers
    fig.add_trace(go.Scatter(
        x=lags[1:], y=vals[1:], mode="markers",
        marker=dict(color=col_color, size=7),
        hovertemplate=f"lag=%{{x}}<br>{title_abbr}=%{{y:.3f}}<extra></extra>",
        showlegend=False,
    ), row=1, col=row_i)
    fig.add_hline(y=0, line_color="#444", line_width=1, row=1, col=row_i)

fig.update_xaxes(title_text="Lag (weeks)")
fig.update_yaxes(title_text="Correlation")
fig.update_layout(title="ACF and PACF on first-differenced series",
    paper_bgcolor="white", plot_bgcolor="white", width=1000, height=420)
fig.show()

In [38]:
# Identify the first lag where ACF / PACF cuts off
sig_thresh = 1.96 / np.sqrt(len(ts_diff1))
acf_cutoff  = next((i for i, v in enumerate(acf_vals[1:],  1) if abs(v) < sig_thresh), n_lags)
pacf_cutoff = next((i for i, v in enumerate(pacf_vals[1:], 1) if abs(v) < sig_thresh), n_lags)

box("output", "Manual reading of ACF and PACF",
    f"Significance threshold (±1.96/√n) = ±{sig_thresh:.3f}<br>"
    f"<b>ACF</b> first enters the band at lag {acf_cutoff} → suggests <b>MA(q) order ≈ {max(1, acf_cutoff-1)}</b> (MA cuts off in ACF).<br>"
    f"<b>PACF</b> first enters the band at lag {pacf_cutoff} → suggests <b>AR(p) order ≈ {max(1, pacf_cutoff-1)}</b> (AR cuts off in PACF).<br><br>"
    "<b>Decision rule reminder:</b> "
    "AR(p) → PACF cuts off at lag p, ACF tails off. "
    "MA(q) → ACF cuts off at lag q, PACF tails off. "
    "ARMA → both tail off gradually.")

In [39]:
box("math", "What ACF and PACF measure",
    "<b>ACF(k)</b> = Corr(y_t, y_{t-k}) — the raw correlation between the series and its k-step lag, "
    "including all indirect effects through intermediate lags.<br>"
    "<b>PACF(k)</b> = the correlation between y_t and y_{t-k} after removing the linear effect of "
    "y_{t-1}, y_{t-2}, ..., y_{t-k+1}. It isolates the <i>direct</i> lag-k relationship.<br><br>"
    "The grey band is the 95% confidence interval under H₀: all autocorrelations are zero. "
    "Bars that poke outside the band are statistically significant.")

---

## Section 4 — ARIMA: From Manual Order to AIC Grid Search

**Teaching pattern in use: Step-by-Step Manual Fit Before the Systematic Search.**

We first fit the model suggested by the ACF/PACF plots, then systematically search over (p, d, q) using AIC to find the optimal order.

In [67]:
# Step 1: fit the manually suggested order
TRAIN_END   = 130   # first 130 weeks for training
TEST_START  = 130
ts_train = ts.iloc[:TRAIN_END]
ts_test  = ts.iloc[TEST_START:]

manual_order = (1, 1, 1)   # ARIMA(1,1,1) as a sensible default from ACF/PACF
model_manual = ARIMA(ts_train, order=manual_order, seasonal_order=(1, 1, 1, 52)).fit()

print(model_manual.summary().tables[0])
print()
print(f"AIC  = {model_manual.aic:.2f}")
print(f"BIC  = {model_manual.bic:.2f}")

                                    SARIMAX Results                                     
Dep. Variable:                     weekly_sales   No. Observations:                  130
Model:             ARIMA(1, 1, 1)x(1, 1, 1, 52)   Log Likelihood                -341.660
Date:                          Sat, 27 Jun 2026   AIC                            693.320
Time:                                  18:59:49   BIC                            705.039
Sample:                              01-04-2021   HQIC                           698.008
                                   - 06-26-2023                                         
Covariance Type:                            opg                                         

AIC  = 693.32
BIC  = 705.04


In [68]:
box("math", "The ARIMA(p, d, q) model",
    "ARIMA = AutoRegressive Integrated Moving Average.<br><br>"
    "<b>AR(p):</b> y_t = φ₁y_{t-1} + ... + φₚy_{t-p} + ε_t  — today depends on the last p values.<br>"
    "<b>I(d):</b> difference the series d times to achieve stationarity.<br>"
    "<b>MA(q):</b> ε_t = θ₁ε_{t-1} + ... + θ_qε_{t-q} + w_t  — the error today depends on the last q errors.<br><br>"
    "Combined: ARIMA fits on the d-th differenced series, using both lagged values (AR) and lagged errors (MA). "
    "The model is estimated by maximum likelihood. <b>AIC = −2·logL + 2k</b> penalises complexity (k = number of parameters).")

In [69]:
# Step 2: AIC grid search over p in {0,1,2} and q in {0,1,2}, d fixed at 1
results_grid = []
for p, q in itertools.product(range(3), range(3)):
    try:
        m = ARIMA(ts_train, order=(p, 1, q), seasonal_order=(p, 1, q, 52)).fit()
        results_grid.append({"order": f"({p},1,{q})", "p": p, "q": q,
                              "AIC": m.aic, "BIC": m.bic})
    except Exception:
        pass

grid_df = (pd.DataFrame(results_grid)
             .sort_values("AIC")
             .reset_index(drop=True))
print("AIC grid search results (sorted by AIC):")
print(grid_df.to_string(index=False))

d:\Languages\Python\Collab-Notebooks\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Languages\Python\Collab-Notebooks\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Languages\Python\Collab-Notebooks\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Languages\Python\Collab-Notebooks\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Languages\Python\Collab-N

AIC grid search results (sorted by AIC):
  order  p  q        AIC        BIC
(0,1,1)  0  1 689.692740 696.724156
(0,1,2)  0  2 693.317396 705.036423
(1,1,1)  1  1 693.320161 705.039188
(2,1,1)  2  1 696.647923 713.054561
(1,1,2)  1  2 696.840181 713.246819
(2,1,2)  2  2 700.566760 721.661009
(2,1,0)  2  0 708.854344 720.573371
(1,1,0)  1  0 712.064814 719.096230
(0,1,0)  0  0 747.410009 749.753814


In [70]:
# Best order from grid
best_row_g = grid_df.iloc[0]
best_p, best_q = int(best_row_g["p"]), int(best_row_g["q"])
BEST_ORDER = (best_p, 1, best_q)
model_best = ARIMA(ts_train, order=BEST_ORDER, seasonal_order=(best_p, 1, best_q, 52)).fit()

box("output", f"AIC grid search winner: ARIMA{BEST_ORDER}",
    f"Best AIC = <b>{best_row_g['AIC']:.2f}</b> for order {BEST_ORDER}.<br>"
    f"Manual ARIMA(1,1,1) AIC was {model_manual.aic:.2f}.<br>"
    f"AIC improvement: {model_manual.aic - best_row_g['AIC']:.2f} units. "
    "Positive = the grid found a genuinely better model. "
    "Negative = the manual guess was already optimal. Either is possible.")

In [71]:
# Forecast on the test window with confidence intervals
forecast_obj  = model_best.get_forecast(steps=len(ts_test))
forecast_mean = forecast_obj.predicted_mean
forecast_ci   = forecast_obj.conf_int()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ts_train.index, y=ts_train.values, mode="lines",
    line=dict(color=BLUE, width=1.5), name="train",
    hovertemplate="week=%{x|%Y-%m-%d}<br>sales=%{y:.1f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=ts_test.index, y=ts_test.values, mode="lines",
    line=dict(color=GREEN, width=2), name="actual test",
    hovertemplate="week=%{x|%Y-%m-%d}<br>actual=%{y:.1f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=forecast_mean.index, y=forecast_mean.values, mode="lines",
    line=dict(color=MASAI_RED, width=2, dash="dash"), name=f"ARIMA{BEST_ORDER} forecast",
    hovertemplate="week=%{x|%Y-%m-%d}<br>forecast=%{y:.1f}<extra></extra>",
))
# 95% CI shading
ci_x = list(forecast_ci.index) + list(forecast_ci.index[::-1])
ci_y = list(forecast_ci.iloc[:, 1].values) + list(forecast_ci.iloc[:, 0].values[::-1])
fig.add_trace(go.Scatter(
    x=ci_x, y=ci_y,
    fill="toself", fillcolor="rgba(237,28,36,0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="95% CI",
    hoverinfo="skip",
))
fig.add_vline(x=ts_train.index[-1].timestamp()*1000, line_dash="dot", line_color=GREY,
              annotation_text="train/test split")
fig.update_layout(
    title=f"ARIMA{BEST_ORDER} forecast vs actuals (26-week horizon)",
    xaxis_title="Date", yaxis_title="Units sold",
    paper_bgcolor="white", plot_bgcolor="white", width=980, height=460,
)
fig.show()

In [72]:
# Residual check: Ljung-Box test for autocorrelation in residuals
resid_arima  = model_best.resid
lb_test      = acorr_ljungbox(resid_arima, lags=[10], return_df=True)
lb_pvalue    = lb_test["lb_pvalue"].iloc[0]

mae_arima  = mean_absolute_error(ts_test.values, forecast_mean.values)
rmse_arima = np.sqrt(mean_squared_error(ts_test.values, forecast_mean.values))
mape_arima = np.mean(np.abs((ts_test.values - forecast_mean.values) / ts_test.values)) * 100

box("output", f"ARIMA{BEST_ORDER} test-set diagnostics",
    f"MAE  = <b>{mae_arima:.2f}</b> units<br>"
    f"RMSE = <b>{rmse_arima:.2f}</b> units<br>"
    f"MAPE = <b>{mape_arima:.2f}%</b><br>"
    f"Ljung-Box p (10 lags) = {lb_pvalue:.4f} "
    f"({'residuals are white noise ✓' if lb_pvalue > 0.05 else 'residuals still autocorrelated — try higher p or q'})")

---

## Section 5 — Time-Feature Engineering (the Prophet ingredient)

**Teaching pattern in use: Manual Construction Before the Framework Call.**

Prophet (Meta's open-source forecaster) works by building a design matrix from calendar and Fourier features, then fitting a linear model. We do exactly that by hand first — so when we call the equivalent in Section 6, you know what is inside it.

In [74]:
# Build the full time-feature matrix from scratch
df_feat = pd.DataFrame({"ds": ts.index, "y": ts.values}).reset_index(drop=True)
df_feat["t"]           = np.arange(len(df_feat))               # linear time index
df_feat["week_of_year"] = df_feat["ds"].dt.isocalendar().week.astype(int)
df_feat["month"]        = df_feat["ds"].dt.month
df_feat["quarter"]      = df_feat["ds"].dt.quarter
df_feat["is_q4"]        = (df_feat["quarter"] == 4).astype(float)

# Fourier terms: encode periodic seasonality without dummy variables
# Period P=52 weeks; K harmonics captures different 'shapes' of the wave
P = 52
for k in range(1, 4):                          # K=3 harmonics
    df_feat[f"sin_{k}"]  = np.sin(2 * np.pi * k * df_feat["t"] / P)
    df_feat[f"cos_{k}"]  = np.cos(2 * np.pi * k * df_feat["t"] / P)

print(f"Feature matrix shape: {df_feat.shape}")
print("Columns:", df_feat.columns.tolist())
df_feat.head(4)

Feature matrix shape: (156, 13)
Columns: ['ds', 'y', 't', 'week_of_year', 'month', 'quarter', 'is_q4', 'sin_1', 'cos_1', 'sin_2', 'cos_2', 'sin_3', 'cos_3']


,ds,y,t,week_of_year,month,quarter,is_q4,sin_1,cos_1,sin_2,cos_2,sin_3,cos_3
0,2021-01-04,505.5,0,1,1,1,0.0,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
1,2021-01-11,505.0,1,2,1,1,0.0,0.120537,0.992709,0.239316,0.970942,0.354605,0.935016
2,2021-01-18,558.2,2,3,1,1,0.0,0.239316,0.970942,0.464723,0.885456,0.663123,0.748511
3,2021-01-25,577.6,3,4,1,1,0.0,0.354605,0.935016,0.663123,0.748511,0.885456,0.464723


In [75]:
box("math", "Why Fourier features encode seasonality",
    "Any periodic function of period P can be approximated as a sum of sine and cosine waves:<br>"
    "S(t) ≈ Σ<sub>k=1..K</sub> [ a_k · sin(2πkt/P) + b_k · cos(2πkt/P) ]<br><br>"
    "With K=1 you get a simple sinusoidal wave. With K=3 you can fit asymmetric shapes "
    "(sharp festive-season spikes, slow recovery). "
    "The amplitudes a_k, b_k are learned by OLS — they are just regression coefficients. "
    "This is exactly what Prophet does: it calls these the 'seasonality component' and fits them jointly with the trend.")

In [76]:
# Visualise how individual Fourier terms look
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Sine harmonics (shape)", "Cosine harmonics (phase shift)"))
for k in range(1, 4):
    fig.add_trace(go.Scatter(
        x=df_feat["ds"], y=df_feat[f"sin_{k}"],
        mode="lines", name=f"sin k={k}",
        hovertemplate=f"sin_{k}=%{{y:.3f}}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df_feat["ds"], y=df_feat[f"cos_{k}"],
        mode="lines", name=f"cos k={k}",
        hovertemplate=f"cos_{k}=%{{y:.3f}}<extra></extra>",
    ), row=1, col=2)
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Feature value")
fig.update_layout(title="Fourier features for annual seasonality (P=52 weeks, K=3 harmonics)",
    paper_bgcolor="white", plot_bgcolor="white", width=980, height=400)
fig.show()

In [77]:
box("tip", "How many Fourier harmonics (K) to use",
    "<b>K=1:</b> one smooth sinusoidal wave — good for slowly-varying seasonality.<br>"
    "<b>K=3–5:</b> can fit asymmetric or peaked seasonal shapes — correct for festive-spike data like ours.<br>"
    "<b>K too large:</b> overfits the seasonal shape to the training noise; the wave wiggles through every small bump.<br>"
    "Rule of thumb: start at K=2–3, increase only if the residuals from Section 1 still show clear seasonal structure.")

---

## Section 6 — Prophet-style Forecasting (Trend + Fourier Seasonality)

**Teaching pattern in use: Build the Framework's Internal Model Yourself.**

We fit a linear regression on the time features we just engineered. This is the core of what Prophet does (it adds changepoints and holiday regressors on top, but the architecture is the same).

In [78]:
# Assemble feature columns for the Prophet-style model
FEATURE_COLS = ["t", "is_q4", "sin_1", "cos_1", "sin_2", "cos_2", "sin_3", "cos_3"]

# Same train/test split as ARIMA
df_train_p = df_feat.iloc[:TRAIN_END].copy()
df_test_p  = df_feat.iloc[TEST_START:].copy()

X_train_p = df_train_p[FEATURE_COLS].values
y_train_p = df_train_p["y"].values
X_test_p  = df_test_p[FEATURE_COLS].values
y_test_p  = df_test_p["y"].values

lr_prophet = LinearRegression().fit(X_train_p, y_train_p)
y_pred_train_p = lr_prophet.predict(X_train_p)
y_pred_test_p  = lr_prophet.predict(X_test_p)

train_r2 = lr_prophet.score(X_train_p, y_train_p)
test_r2  = lr_prophet.score(X_test_p, y_test_p)
print(f"Train R² = {train_r2:.4f}")
print(f"Test  R² = {test_r2:.4f}")

Train R² = 0.9645
Test  R² = 0.4357


In [79]:
# Interpret the learned coefficients
coef_df = pd.DataFrame({
    "feature":     FEATURE_COLS,
    "coefficient": lr_prophet.coef_,
}).sort_values("coefficient", key=abs, ascending=False).reset_index(drop=True)

fig = go.Figure(go.Bar(
    x=coef_df["coefficient"], y=coef_df["feature"], orientation="h",
    marker_color=[MASAI_RED if c > 0 else BLUE for c in coef_df["coefficient"]],
    text=[f"{c:.2f}" for c in coef_df["coefficient"]], textposition="outside",
    hovertemplate="<b>%{y}</b><br>coef = %{x:.3f}<extra></extra>",
))
fig.update_layout(
    title="Prophet-style model: learned feature coefficients",
    xaxis_title="Coefficient (units per unit of feature)",
    yaxis=dict(autorange="reversed"),
    paper_bgcolor="white", plot_bgcolor="white", width=820, height=420,
)
fig.show()

In [82]:
# Recover trend and seasonality from the model
pred_components = {
    "trend":     lr_prophet.intercept_ + lr_prophet.coef_[0] * df_feat["t"],
    "q4_effect": lr_prophet.coef_[1] * df_feat["is_q4"],
    "fourier":   sum(lr_prophet.coef_[i+2] * df_feat[col]
                     for i, col in enumerate(["sin_1","cos_1","sin_2","cos_2","sin_3","cos_3"])),
}

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=("Learned Trend", "Fourier Seasonal (K=3)", "Q4 Holiday Effect"),
    vertical_spacing=0.09)
colors_c = [MASAI_RED, BLUE, ORANGE]
labels_c = ["trend", "fourier", "q4_effect"]
for i, (lab, col) in enumerate(zip(labels_c, colors_c), 1):
    fig.add_trace(go.Scatter(x=df_feat["ds"], y=pred_components[lab].values,
        mode="lines", line=dict(color=col, width=1.8),
        hovertemplate=f"week=%{{x|%Y-%m-%d}}<br>{lab}=%{{y:.2f}}<extra></extra>",
        showlegend=False), row=i, col=1)
fig.update_layout(title="Decomposed components from the Prophet-style model",
    paper_bgcolor="white", plot_bgcolor="white", width=950, height=600)
fig.show()

In [83]:
# Side-by-side: ARIMA vs Prophet-style on test window
mae_p  = mean_absolute_error(y_test_p, y_pred_test_p)
rmse_p = np.sqrt(mean_squared_error(y_test_p, y_pred_test_p))
mape_p = np.mean(np.abs((y_test_p - y_pred_test_p) / y_test_p)) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test.index, y=ts_test.values, mode="lines",
    line=dict(color=GREEN, width=2), name="actual",
    hovertemplate="week=%{x|%Y-%m-%d}<br>actual=%{y:.1f}<extra></extra>"))
fig.add_trace(go.Scatter(x=ts_test.index, y=forecast_mean.values, mode="lines",
    line=dict(color=MASAI_RED, width=2, dash="dash"), name=f"ARIMA{BEST_ORDER}",
    hovertemplate="week=%{x|%Y-%m-%d}<br>ARIMA=%{y:.1f}<extra></extra>"))
fig.add_trace(go.Scatter(x=df_test_p["ds"], y=y_pred_test_p, mode="lines",
    line=dict(color=PURPLE, width=2, dash="dot"), name="Prophet-style",
    hovertemplate="week=%{x|%Y-%m-%d}<br>Prophet=%{y:.1f}<extra></extra>"))
fig.update_layout(
    title="ARIMA vs Prophet-style model — test window",
    xaxis_title="Date", yaxis_title="Units sold",
    paper_bgcolor="white", plot_bgcolor="white", width=980, height=440,
)
fig.show()

In [85]:
box("output", "ARIMA vs Prophet-style: test-set metrics",
    f"<b>ARIMA{BEST_ORDER}</b>: MAE={mae_arima:.2f}, RMSE={rmse_arima:.2f}, MAPE={mape_arima:.2f}%<br>"
    f"<b>Prophet-style</b>: MAE={mae_p:.2f}, RMSE={rmse_p:.2f}, MAPE={mape_p:.2f}%<br><br>"
    "Which wins depends on the horizon. ARIMA uses recent autocorrelation — it is typically better at 1–4 step-ahead. "
    "The Prophet-style model uses the full seasonal structure — it extrapolates further without degrading. "
    "Walk-forward validation in Section 7 will reveal this trade-off honestly.")

In [86]:
box("definition", "What Prophet actually adds on top",
    "The model we built above is Prophet's skeleton. The full Prophet library (Meta, 2017) adds:<br>"
    "<b>1. Changepoint detection:</b> allows the trend slope to change at automatically detected dates "
    "(e.g. when a competitor launched, or lockdown started). We assumed a constant slope.<br>"
    "<b>2. Holiday regressors:</b> binary features for specific calendar events — same as our is_q4 but for individual holidays.<br>"
    "<b>3. Stan-based MCMC/MAP fitting:</b> provides full posterior uncertainty rather than OLS point estimates.<br><br>"
    "If you understand this notebook, you understand the ingredient list. Prophet is a convenience wrapper "
    "that automates changepoint selection and holiday encoding — it is not magic.")

---

## Section 7 — Walk-Forward (Expanding-Window) Validation

**Teaching pattern in use: Visual Evidence of Why Holdout Evaluation Misleads.**

We compare a single holdout evaluation (train/test split) against walk-forward validation. Walk-forward simulates production use: fit on all data up to time t, predict step t+1, advance, repeat.

In [87]:
box("definition", "Walk-forward (expanding-window) validation",
    "For each step i in {train_size, train_size+1, ..., N-1}:<br>"
    "&nbsp;&nbsp;1. Fit the model on ts[0 : i] (all data seen so far).<br>"
    "&nbsp;&nbsp;2. Forecast one step ahead: ŷ[i].<br>"
    "&nbsp;&nbsp;3. Record the error: e[i] = y[i] - ŷ[i].<br>"
    "&nbsp;&nbsp;4. Advance i by 1.<br><br>"
    "This gives a realistic error distribution: at each step the model has exactly the same information "
    "it would have in production. A single holdout split conflates early-horizon accuracy (easy) "
    "with late-horizon accuracy (hard) into one number.")

In [91]:
# Walk-forward validation for ARIMA (1-step-ahead)
WF_START = TRAIN_END
actuals_wf, preds_arima_wf, preds_prophet_wf, wf_dates = [], [], [], []

for i in range(WF_START, N):
    # --- ARIMA ---
    try:
        m_a = ARIMA(ts.iloc[:i], order=BEST_ORDER).fit()
        fc_a = m_a.forecast(steps=1).iloc[0]
    except Exception:
        fc_a = ts.iloc[i-1]   # fallback: last observed

    # --- Prophet-style ---
    X_wf = df_feat.iloc[:i][FEATURE_COLS].values
    y_wf = df_feat.iloc[:i]["y"].values
    lr_wf = LinearRegression().fit(X_wf, y_wf)
    fc_p = lr_wf.predict(df_feat.iloc[[i]][FEATURE_COLS].values)[0]

    actuals_wf.append(ts.iloc[i])
    preds_arima_wf.append(fc_a)
    preds_prophet_wf.append(fc_p)
    wf_dates.append(ts.index[i])

actuals_wf      = np.array(actuals_wf)
preds_arima_wf  = np.array(preds_arima_wf)
preds_prophet_wf = np.array(preds_prophet_wf)
print(f"Walk-forward steps completed: {len(actuals_wf)}")

Walk-forward steps completed: 26


In [92]:
# Compute rolling MAE to see how accuracy evolves over time
window = 8
roll_mae_arima   = pd.Series(np.abs(actuals_wf - preds_arima_wf)).rolling(window).mean()
roll_mae_prophet = pd.Series(np.abs(actuals_wf - preds_prophet_wf)).rolling(window).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Walk-forward: actual vs predictions",
                    f"Rolling {window}-week MAE (lower = better)"),
    vertical_spacing=0.12)

fig.add_trace(go.Scatter(x=wf_dates, y=actuals_wf, mode="lines",
    line=dict(color=GREEN, width=2), name="actual",
    hovertemplate="week=%{x|%Y-%m-%d}<br>actual=%{y:.1f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=wf_dates, y=preds_arima_wf, mode="lines",
    line=dict(color=MASAI_RED, width=1.8, dash="dash"), name=f"ARIMA{BEST_ORDER}",
    hovertemplate="week=%{x|%Y-%m-%d}<br>ARIMA=%{y:.1f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=wf_dates, y=preds_prophet_wf, mode="lines",
    line=dict(color=PURPLE, width=1.8, dash="dot"), name="Prophet-style",
    hovertemplate="week=%{x|%Y-%m-%d}<br>Prophet=%{y:.1f}<extra></extra>"), row=1, col=1)

fig.add_trace(go.Scatter(x=wf_dates, y=roll_mae_arima, mode="lines",
    line=dict(color=MASAI_RED, width=2), name=f"ARIMA MAE ({window}w rolling)",
    showlegend=True,
    hovertemplate="week=%{x|%Y-%m-%d}<br>rolling MAE=%{y:.2f}<extra></extra>"), row=2, col=1)
fig.add_trace(go.Scatter(x=wf_dates, y=roll_mae_prophet, mode="lines",
    line=dict(color=PURPLE, width=2), name=f"Prophet MAE ({window}w rolling)",
    showlegend=True,
    hovertemplate="week=%{x|%Y-%m-%d}<br>rolling MAE=%{y:.2f}<extra></extra>"), row=2, col=1)

fig.update_yaxes(title_text="Units sold", row=1, col=1)
fig.update_yaxes(title_text="MAE (units)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_layout(title="Walk-forward validation: accuracy over time",
    paper_bgcolor="white", plot_bgcolor="white", width=980, height=600)
fig.show()

In [93]:
# Summary table
mae_wf_a    = mean_absolute_error(actuals_wf, preds_arima_wf)
rmse_wf_a   = np.sqrt(mean_squared_error(actuals_wf, preds_arima_wf))
mape_wf_a   = np.mean(np.abs((actuals_wf - preds_arima_wf) / actuals_wf)) * 100
mae_wf_p    = mean_absolute_error(actuals_wf, preds_prophet_wf)
rmse_wf_p   = np.sqrt(mean_squared_error(actuals_wf, preds_prophet_wf))
mape_wf_p   = np.mean(np.abs((actuals_wf - preds_prophet_wf) / actuals_wf)) * 100

summary = pd.DataFrame({
    "Model":        [f"ARIMA{BEST_ORDER}", "Prophet-style"],
    "Walk-fwd MAE": [f"{mae_wf_a:.2f}",   f"{mae_wf_p:.2f}"],
    "Walk-fwd RMSE":[f"{rmse_wf_a:.2f}",  f"{rmse_wf_p:.2f}"],
    "Walk-fwd MAPE":[f"{mape_wf_a:.2f}%", f"{mape_wf_p:.2f}%"],
})
print(summary.to_string(index=False))

winner = f"ARIMA{BEST_ORDER}" if mae_wf_a < mae_wf_p else "Prophet-style"
box("output", "Walk-forward validation verdict",
    f"<b>{winner}</b> wins on MAE under 1-step-ahead walk-forward validation.<br>"
    f"ARIMA walk-fwd MAE = {mae_wf_a:.2f} vs holdout MAE = {mae_arima:.2f}. "
    f"Prophet walk-fwd MAE = {mae_wf_p:.2f} vs holdout MAE = {mae_p:.2f}.<br><br>"
    "<b>Key insight:</b> holdout and walk-forward can give different rankings. "
    "Always report the walk-forward number — it is the one that reflects production conditions.")

         Model Walk-fwd MAE Walk-fwd RMSE Walk-fwd MAPE
ARIMA(0, 1, 1)        20.21         25.04         2.50%
 Prophet-style        21.00         25.57         2.62%


In [94]:
box("danger", "Why a single holdout split misleads in time series",
    "With random CV (used for cross-sectional data), each fold sees the full temporal range. "
    "For time series, that is data leakage: a model trained on 2023 data should never be tested on 2022 data it hasn't seen yet.<br><br>"
    "Walk-forward validation respects the temporal order. Each prediction is made using only data available at that point in time. "
    "The rolling MAE plot shows something a single number hides: whether the model is improving as it gets more data, "
    "or degrading as it enters unfamiliar territory (e.g. post-festive-season correction).")

---

## Mini-Project — Retail Store Revenue Forecasting

You are given a synthetic weekly revenue dataset for a grocery chain. Your job is to run the full forecasting pipeline: decompose, test stationarity, fit ARIMA, build a Prophet-style model, evaluate both with walk-forward validation, and write a 3-bullet deployment assessment.

In [95]:
# Mini-project dataset — 2.5 years of weekly grocery revenue
M       = 130
rng_mp  = np.random.default_rng(SEED + 99)
t_mp    = np.arange(M)
dates_mp = pd.date_range("2022-01-03", periods=M, freq="W-MON")
revenue = (
    8000
    + 3.0 * t_mp
    + 600 * np.sin(2 * np.pi * t_mp / 52)          # annual cycle
    + 200 * np.sin(2 * np.pi * t_mp / 13)          # quarterly
    + 400 * (t_mp > 95).astype(float)              # step change: new store opened
    + rng_mp.normal(0, 120, M)
).round(0)

ts_mp = pd.Series(revenue, index=dates_mp, name="weekly_revenue")
print(f"Mini-project series shape: {ts_mp.shape}")
ts_mp.describe().round(1)

Mini-project series shape: (130,)


count     130.0
mean     8362.4
std       560.1
min      7154.0
25%      7946.2
50%      8381.5
75%      8706.5
max      9866.0
Name: weekly_revenue, dtype: float64

**Your Tasks** (replace each `pass` with working code):

**Task 1** — Decompose `ts_mp` (additive, period=52). Plot all four panels.

**Task 2** — Run ADF on the raw series. If non-stationary, apply first differencing and confirm stationarity.

**Task 3** — Plot ACF and PACF on the differenced series. Write a comment stating your manual (p, q) guess.

**Task 4** — Run an AIC grid search over p ∈ {0,1,2}, d=1, q ∈ {0,1,2}. Fit the best ARIMA model.

**Task 5** — Engineer Fourier features (K=3, P=52) and fit the Prophet-style linear model.

**Task 6** — Run walk-forward validation (train on first 100 weeks) for both models. Report MAE, RMSE, MAPE.

**Task 7** — In the markdown cell below, write your 3-bullet deployment assessment.

In [96]:
# YOUR CODE — Task 1: Decompose ts_mp
# decomp_mp = seasonal_decompose(...)
# fig = make_subplots(...)
decomp_mp = seasonal_decompose(ts_mp, model="additive", period=52)
print("Mini-project decomposition complete. Components:")
print(f"  Trend shape: {decomp_mp.trend.shape}")
print(f"  Seasonal shape: {decomp_mp.seasonal.shape}")
print(f"  Residual shape: {decomp_mp.resid.shape}")


# Self-check (uncomment when ready):
# assert decomp_mp.trend.dropna().shape[0] > 0, "Decomposition failed"

Mini-project decomposition complete. Components:
  Trend shape: (130,)
  Seasonal shape: (130,)
  Residual shape: (130,)


In [102]:
# YOUR CODE — Task 2: ADF test + differencing
# adf_raw_mp = adfuller(...)
# ts_diff_mp = ts_mp.diff().dropna()
# adf_diff_mp = adfuller(...)
adf, p_val, _, _, _, _ = adfuller(ts_mp)

if p_val < 0.05:
    print(f"Mini-project series is stationary (ADF p = {p_val:.4f})")
else:
    print(f"Mini-project series is non-stationary (ADF p = {p_val:.4f}) — differencing required")
    ts_diff_mp = ts_mp.diff().dropna()
    adf_diff, p_val_diff = adfuller(ts_diff_mp)
    if p_val_diff < 0.05:
        print(f"Differenced series is stationary (ADF p = {p_val_diff:.4f})")
    else:
        print(f"Differenced series is still non-stationary (ADF p = {p_val_diff:.4f})")

# Self-check:
# assert adf_diff_mp[1] < 0.05, "Differenced series should be stationary"

Mini-project series is stationary (ADF p = 0.0025)


In [43]:
# YOUR CODE — Task 3: ACF and PACF plot
# acf_vals_mp, ... = acf(ts_diff_mp, ...)
# pacf_vals_mp, ... = pacf(...)
# ... plot ...
# Manual guess: p = ?, q = ?
pass

In [44]:
# YOUR CODE — Task 4: AIC grid search → best ARIMA
# results_mp = []
# for p, q in itertools.product(...):
#     ...
# best_order_mp = ...
pass

# Self-check:
# assert best_order_mp[1] == 1, "d must be 1 for this series"

In [45]:
# YOUR CODE — Task 5: Fourier features + Prophet-style model
# df_mp = pd.DataFrame({"ds": ts_mp.index, "y": ts_mp.values}).reset_index(drop=True)
# df_mp["t"] = np.arange(len(df_mp))
# ... add Fourier features ...
# lr_mp = LinearRegression().fit(...)
pass

In [46]:
# YOUR CODE — Task 6: Walk-forward validation (first 100 weeks as initial train)
# WF_START_MP = 100
# for i in range(WF_START_MP, M):
#     ...
# Print MAE, RMSE, MAPE for both models
pass

### Task 7 — Your 3-bullet deployment assessment

*(Edit this cell)*

- **Decomposition:** does the residual look like white noise, or is structure missed? ...
- **ARIMA order chosen + Ljung-Box verdict:** ...
- **Walk-forward verdict:** which model wins and by how much? Would you deploy it for a 4-week demand plan? ...

**Final call:** *Yes / No / Not yet — because ...*

---
## Data Quality Dashboard (final asserts)

In [47]:
# Series invariants
assert ts.shape == (N,),                         f"Main series shape: {ts.shape}"
assert ts.isna().sum() == 0,                     "NaN in main series"
assert (ts > 0).all(),                           "All sales should be positive"
assert isinstance(ts.index, pd.DatetimeIndex),   "Index must be DatetimeIndex"

# Stationarity
assert p_raw  > 0.05,                            "Raw series should be non-stationary"
assert p_d1   < 0.05,                            "Differenced series should be stationary"

# ARIMA
assert len(BEST_ORDER) == 3,                     "ARIMA order must be a 3-tuple"
assert BEST_ORDER[1] == 1,                       "d must be 1 for this series"
assert forecast_mean.shape[0] == len(ts_test),   "Forecast length must match test window"

# Feature engineering
assert "sin_1" in df_feat.columns,              "Fourier sin_1 feature missing"
assert "cos_1" in df_feat.columns,              "Fourier cos_1 feature missing"
assert df_feat["t"].iloc[-1] == N - 1,          "Linear time index incorrect"

# Walk-forward
assert len(actuals_wf) == N - TRAIN_END,         "Walk-forward step count mismatch"
assert 0 < mae_wf_a < 500,                       "ARIMA walk-fwd MAE out of plausible range"

# Mini-project dataset
assert ts_mp.shape == (M,),                      f"Mini-project series shape: {ts_mp.shape}"
assert (ts_mp > 0).all(),                        "Revenue should be positive"

print("All data-quality asserts passed ✓")

All data-quality asserts passed ✓


---

## Ten Takeaways

1. A time series has three separable components: trend, seasonality, and residual. Decomposing first reveals which component carries the most variance and which model form (additive vs multiplicative) is appropriate.
2. The ADF test formalises the intuition that a trending series cannot be modelled by ARIMA as-is. The null is a unit root (non-stationary); p < 0.05 means safe to proceed. Almost all macro series are I(1): one difference is enough.
3. ACF measures raw lag correlations including indirect paths; PACF isolates the direct lag-k effect. AR(p) shows PACF cutoff; MA(q) shows ACF cutoff; ARMA shows both tailing off.
4. AIC grid search beats manual order selection because it trades off fit quality against model complexity. Lower AIC = better model, penalised for parameters.
5. Fourier features (sin/cos pairs at harmonics of the fundamental period) encode any periodic seasonal pattern as regression coefficients — no one-hot explosion, no black-box.
6. Prophet is an OLS model on a design matrix of trend, Fourier seasonality, and holiday regressors, fitted by MAP under weakly informative priors. Knowing this you can reproduce most of what it does with a LinearRegression.
7. Walk-forward (expanding-window) validation respects temporal order. Each forecast is made with only historically available data. Use it instead of k-fold for all time series evaluation.
8. A single holdout split can over-estimate accuracy if the test window happens to be an easy period, or under-estimate it if the test window is unusually hard. Walk-forward averages over many windows.
9. ARIMA tends to beat on short horizons (1–4 steps) because it exploits recent autocorrelation. Feature-based models (Prophet-style) tend to win on longer horizons because they extrapolate structural patterns.
10. The Flipkart trap: high in-sample R² with a non-stationary, heteroscedastic series is meaningless. The three diagnostics — stationarity, residual ACF, walk-forward MAE — are what stand between a notebook result and a reliable production forecast.

---

**Where this leads next**: multi-step horizon forecasting, ensemble stacking of ARIMA + gradient boosting on time features, and LSTM-based sequence models for very long-memory series.

---

Vishlesan i-Hub IIT Patna × Masai School